# 第 1 周第 2 天 —— 用本地开源模型做网页摘要

## 练习目标（理念）

抓取网页正文，通过 **Ollama 的 OpenAI 兼容接口**（`/v1`）调用本地模型（如 `llama3.2`、`deepseek-r1:1.5b`），生成 Markdown 摘要。对比不同开源模型的效果。

## 和本课 Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| OpenAI 兼容客户端 | `OpenAI(base_url=..., api_key='ollama')` |
| 本地 Ollama | `http://localhost:11434/v1` |
| 网页抓取 | `fetch_website_contents` |
| 换模型对比 | 同一 URL，分别用 `llama3.2` 与 `deepseek-r1:1.5b` |

## 怎么跑

1. 本机启动 Ollama，并已 `ollama pull` 对应模型
2. 同目录需有可用的 `scraper.py`
3. 从上到下运行；后两格会对 `https://edwarddonner.com` 各摘要一次


In [ ]:
# ========== 导入：抓网页、笔记本展示、OpenAI 兼容客户端 ==========

# 从同目录 scraper 导入：抓取网页正文，供模型摘要
from scraper import fetch_website_contents
# Markdown / display：把模型回复渲染成好看的 Markdown
from IPython.display import Markdown, display
# OpenAI 客户端类：这里既可连云端，也可通过 base_url 连本地 Ollama
from openai import OpenAI


In [ ]:
# ========== 两个客户端：默认 OpenAI + 指向本地 Ollama 的兼容客户端 ==========

# 创建默认 OpenAI 客户端实例（本练习摘要实际走下面的 ollama）
openai = OpenAI()

# Ollama 的 OpenAI 兼容 API 根地址（注意是 /v1，不是原生 /api/chat）
OLLAMA_BASE_URL = "http://localhost:11434/v1"
# 用同一套 OpenAI SDK 连本地：base_url 指向 Ollama；api_key 占位即可（本地常不校验）
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# ========== summarize：抓网页 → 调本地模型 → 展示 Markdown ==========

def summarize(url, model):
    # 抓取目标 URL 的网页正文
    website = fetch_website_contents(url)
    # 通过 ollama 客户端调用 Chat Completions；model 由调用方传入（如 llama3.2）
    # messages 里只有 user：把「请摘要」指令和网页正文拼进同一条 content
    # 提示词英文字符串保持原样，勿改译
    response = ollama.chat.completions.create(model=f"{model}", messages=[{"role": "user", "content": f"Summarize the following website content: {website}"}])
    # 取出回复正文，渲染为 Markdown 并 display；函数返回值是 display 的返回值
    return display(Markdown(response.choices[0].message.content))


In [ ]:
# ========== 实跑 1：用本地 llama3.2 摘要 edwarddonner.com ==========

# URL 与 model id 保持原样；需本机已 pull llama3.2
summarize("https://edwarddonner.com", model="llama3.2")


The website is the personal homepage of Edward Donner, a co-founder and CTO of Nebula.io, an AI startup. The site offers various options for visitors, including:

* A blog-style section where Donner shares updates about his work, personal interests, and expertise in AI.
* Links to posts related to topics such as AI in production, leadership, and executive briefings.
* An arena called "Outsmart," which hosts a competitive match of LLMs (large language models) against each other.

Donner also invites visitors to connect with him through social media or subscribe to his newsletter. The website includes links to Donner's professional profiles on LinkedIn, Twitter, and Facebook, as well as an email address for direct contact.

Overall, the site appears to be a personal platform for Donner to share his thoughts on AI, connect with others interested in the topic, and showcase his expertise and experience.

In [ ]:
# ========== 实跑 2：换用 deepseek-r1:1.5b，对比开源模型差异 ==========

# 同一网页、不同模型；便于观察风格/质量差异
summarize("https://edwarddonner.com", model="deepseek-r1:1.5b")


Edward Donner, CEO and CTO of nebula.io, highlights an AI-powered platform offering applications in talent sourcing, with event dates formatted as separate sections linked to the AI event calendar at [link](https://www.edwarddonner.com). 

Key features include an arena where LLMs compete strategically, supported by educational content such as Ed's blog and YouTube channel. The company emphasizes its impact on fostering individuals' potential through its AI applications.

For more details, visit [nebula.io](http://nebula.io) or contact info available on the website. Stay tuned for live events at specific dates, including Connect Four, Outsmart, and others.